<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.1-spm-and-thermal/Ex10.1_01_spm_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_10.1 · Notebook 01 — the single particle

**Paired with L10.1 · Battery models**

Solve the dimensionless particle-diffusion problem with a PINN and compare
against the analytic solution. This is the only physics you write from scratch;
notebooks 02 and 03 build on it.

## What you will do

1. Write the radial residual, and decide what to do about the centre.
2. Assemble the three-condition loss — two of the three are on the *flux*.
3. Train, and look at the surface error separately from the bulk error.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.1-spm-and-thermal/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The residual

$$\mathcal{F} = \frac{\partial c}{\partial t}
- \frac{\partial^2 c}{\partial r^2}
- \frac{2}{r}\frac{\partial c}{\partial r}$$

Watch the $2/r$ term at the centre. Sample $r$ away from zero, or handle the
singularity explicitly — this is a real numerical trap.

`rt` is a tensor of collocation points made with
`to_tensor(pb.particle_points(...), requires_grad=True)`: column 0 is $r$,
column 1 is $t$, exactly as in notebook 00.

### Your turn

In [ ]:
# TODO 1 --- the radial diffusion residual --------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  d2(c, rt, 0)                                                     c_rr
#   line 2  ->  c_t - c_rr - 2.0 / torch.clamp(r, min=pb.R_CENTRE_EPS) * c_r     c_t = c_rr + (2/r) c_r
# The clamp is one way to survive the centre; sampling with r_min=pb.R_CENTRE_EPS is another.
def residual_fn(model, rt):
    c = model(rt)
    g = grad(c, rt)
    c_r, c_t = g[:, 0:1], g[:, 1:2]
    r = rt[:, 0:1]
    c_rr = ...                                    # <- d2(c, rt, 0)
    return ...                                    # <- c_t - c_rr - 2.0 / torch.clamp(r, min=pb.R_CENTRE_EPS) * c_r
# ------------------------------------------------------------------------------

## 2 · The loss

Three conditions: zero flux at the centre, unit flux at the surface, and
$c=0$ initially. Consider hard-enforcing the initial condition with the
$(1-t)$ trick from L8.2 — it removes a term and the causality risk with it.

`problem.py` supplies the three point sets, because each edge of this
rectangle means something different: `pb.particle_surface_points`,
`pb.particle_centre_points`, `pb.particle_initial_points`. All three return
NumPy; `to_tensor` puts them on the device, and only the two flux sets need
`requires_grad=True`.

### Your turn

In [ ]:
# TODO 2 --- the loss: physics, two flux conditions, initial state ------------------------------------
# Three `...` to replace:
#   line 1  ->  mse(grad(model(r_s), r_s)[:, 0:1] - 1.0)        surface flux c_r = 1 (the current arriving)
#   line 2  ->  mse(grad(model(r_0), r_0)[:, 0:1])              centre symmetry c_r = 0
#   line 3  ->  mse(model(ic))                                  initially c = 0
def loss_fn_factory(model, rt):
    n = 200
    r_s = to_tensor(pb.particle_surface_points(n, t_end=1.0), requires_grad=True)   # r = 1
    r_0 = to_tensor(pb.particle_centre_points(n, t_end=1.0), requires_grad=True)    # r = eps
    ic  = to_tensor(pb.particle_initial_points(n))                                  # t = 0, no grad needed

    def loss_fn():                                # no arguments: the contract with train_two_stage
        L_pde  = mse(residual_fn(model, rt))
        L_surf = ...                              # <- mse(grad(model(r_s), r_s)[:, 0:1] - 1.0)
        L_cen  = ...                              # <- mse(grad(model(r_0), r_0)[:, 0:1])
        L_ic   = ...                              # <- mse(model(ic))
        return L_pde + 10.0 * (L_surf + L_cen + L_ic)
    return loss_fn
# ------------------------------------------------------------------------------

## 3 · Train

`pb.run_particle` builds the network, draws the collocation points, calls
`train_two_stage` and scores the result against `pb.analytic_sphere` at five
instants. It prints the collocation-to-parameter ratio first: fewer points than
parameters means the residual can be satisfied everywhere you looked and
anything at all in between.

In [ ]:
cell = pb.CellParams(c_rate=1.0)
result = pb.run_particle(cell, residual_fn, loss_fn_factory,
                         n_coll=3000, n_hidden=32, n_layers=4)
pb.plot_particle(result)

plot_curves(result["history"], title="the single particle — Adam, then L-BFGS")
plt.show()

## 4 · Save

In [ ]:
import pickle
os.makedirs("Ex10.1_outputs", exist_ok=True)
path = os.path.join("Ex10.1_outputs", "ex101_spm.pkl")
with open(path, "wb") as f:
    pickle.dump({k: v for k, v in result.items() if k != "model"}, f)
print("wrote", path)

for e in result["errors"]:
    print(f"  t={e['t']:.2f}  rel {e['rel']:.3e}  "
          f"max {e['max']:.3e}  surface err {e['surf_err']:.3e}")

**Before moving on.** The surface concentration is what the reaction sees, so
the surface error matters more than the bulk error. Is yours worse at the
surface than in the interior? Why would that be?

Next: **notebook 02**, where the model you just built meets a cell that is not
a single particle.